# Laboratorio 5. Naive Bayes

La empresa **SmartStay Advisors** es una firma intermediaria que facilita la búsqueda y selección de
propiedades en Airbnb para clientes corporativos y particulares.
El modelo operativo es el siguiente:

• El cliente proporciona requerimientos específicos (ciudad, presupuesto, tipo de propiedad,
capacidad, duración de estancia).

• SmartStay analiza la oferta disponible y propone propiedades que optimicen precio, calidad
y disponibilidad.

• Airbnb ofrece incentivos económicos a SmartStay cuando logra incrementar el nivel de
ocupación de propiedades con bajo desempeño.
La empresa desea implementar modelos de minería de datos que le permitan:
1. Estimar precios competitivos.
2. Identificar propiedades con baja ocupación.
3. Comprender qué factores influyen en la ocupación y los ingresos.
4. Diseñar estrategias basadas en datos para aumentar la rentabilidad.

## Conjunto de datos a utilizar
listings.RData

In [19]:
import pandas as pd
import pyreadr
resultado = pyreadr.read_r('listings.RData')
print("Objetos contenidos en el archivo RData:", resultado.keys())
df = list(resultado.values())[0]
print(df.shape)

numericas = df.select_dtypes(include=['number']).columns.tolist()
print("=== Variables Numéricas ===")
print(numericas)

categoricas = df.select_dtypes(include=['object']).columns.tolist()
print("=== Variables categoricas ===")
print(categoricas)

limpio = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(limpio, errors='coerce')

Objetos contenidos en el archivo RData: odict_keys(['listings'])
(171748, 80)
=== Variables Numéricas ===
['id', 'scrape_id', 'host_id', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'minimum_nights', 'maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'reviews_per_month']
=== Variables categoricas ===
['listing_url', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview'

C:\Users\richi\AppData\Local\Temp\ipykernel_16776\2377985809.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = df.select_dtypes(include=['object']).columns.tolist()


Usamos el dataset utilizado en laboratorios anteriores, en este se nos da un conjunto de datos con la informaciond de los aribnbs. Los datos obtendios son los siguentes 
| Categorias      | Tipo de Dato   |
| ------------- | ------------     |
| **Numerica**         | 'id', 'scrape_id', 'host_id', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'minimum_nights', 'maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'reviews_per_month'     |
| **Categoricas**     | 'listing_url', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'property_type', 'room_type', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'calendar_updated', 'has_availability', 'calendar_last_scraped', 'estimated_revenue_l365d', 'first_review', 'last_review', 'license', 'instant_bookable', 'city'   |

Convertimos el price de una variable categorica a una numerica para poder hacer un analisis mas profundo y que los modelos puedan predecir de una buena manera


In [17]:
faltantes = df.isnull().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
print("=== CANTIDAD EXACTA DE DATOS VACÍOS ===")
print(faltantes)
print("\n" + "="*40 + "\n")

porcentajes = (faltantes / len(df)) * 100
print("=== PORCENTAJE DE VACÍOS POR COLUMNA ===")
print(porcentajes.round(2).astype(str) + " %")

=== CANTIDAD EXACTA DE DATOS VACÍOS ===
calendar_updated                171748
price                            95502
estimated_revenue_l365d          95502
neighbourhood_group_cleansed     50683
review_scores_value              40328
review_scores_location           40328
review_scores_checkin            40324
review_scores_accuracy           40312
review_scores_communication      40308
review_scores_cleanliness        40302
review_scores_rating             40287
reviews_per_month                40287
beds                             31686
bathrooms                        31396
license                          10533
bedrooms                         10473
host_listings_count                876
host_total_listings_count          876
maximum_maximum_nights              71
maximum_minimum_nights              71
minimum_minimum_nights              71
minimum_maximum_nights              71
host_about                           1
dtype: int64


=== PORCENTAJE DE VACÍOS POR COLUMNA ===
calenda